# UNSUPERVISED LEARNING
## SESSION 3

Assignment questions and answers


## Question 1

Mini Project: Food Delivery Customer Intelligence System

Objective: Build a console-based customer intelligence tool for a food
delivery platform that integrates descriptive statistics (Module 5),
supervised classification (Module 6), and K-Means clustering (Module 7)
into a single menu-driven program — demonstrating end-to-end data
science reasoning across all three modules.

Requirements: - The program must be menu-driven with at least 4 options
displayed at startup: (1) View Statistical Summary, (2) Predict
Cancellation Risk for a New Order, (3) Run Customer Segmentation, (4)
Exit — and must loop back to the menu after each operation until the
user selects Exit. - Option 1 must load a sample in-memory dataset at
startup and print the mean, median, standard deviation, and skewness
direction for at least two numeric columns, with a brief plain-English
interpretation of the skewness finding. - Option 2 must accept
user-typed inputs for order_value, delivery_distance_km, and
restaurant_rating, pass them through a pre-fitted or in-session trained
Random Forest classifier, and display the result as ‘High Cancellation
Risk’ or ‘Low Cancellation Risk’ with the predicted probability. -
Option 3 must apply K-Means (k=3) to customer data loaded at startup,
print the number of customers assigned to each cluster, and display each
cluster’s centroid values rounded to 2 decimal places so the output is
interpretable without a chart. - The program must validate all numeric
user inputs in Option 2, display a descriptive error message for invalid
entries (e.g. non-numeric or out-of-range values), and return to the
menu without crashing.

**Answer:**

```python
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier

# Sample in-memory dataset
data = {
    "order_value": [
        15.0,
        45.0,
        22.5,
        80.0,
        12.0,
        110.0,
        35.0,
        60.0,
        18.0,
        95.0,
        25.0,
        50.0,
    ],
    "delivery_distance_km": [
        2.1,
        8.5,
        3.2,
        12.0,
        1.5,
        15.0,
        5.0,
        9.2,
        2.8,
        14.1,
        4.0,
        7.5,
    ],
    "restaurant_rating": [
        4.5,
        3.2,
        4.8,
        2.5,
        4.1,
        2.1,
        3.8,
        3.0,
        4.9,
        2.8,
        4.2,
        3.5,
    ],
    "is_cancelled": [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0],
}
df = pd.DataFrame(data)

# Pre-train Random Forest Classifier
X = df[["order_value", "delivery_distance_km", "restaurant_rating"]]
y = df["is_cancelled"]
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X, y)


def option_1():
    print("\n--- Statistical Summary ---")
    cols = ["order_value", "delivery_distance_km"]
    for col in cols:
        mean_val = df[col].mean()
        median_val = df[col].median()
        std_val = df[col].std()
        skew_val = df[col].skew()

        direction = (
            "Right (Positive)"
            if skew_val > 0
            else "Left (Negative)"
            if skew_val < 0
            else "Symmetric"
        )

        print(f"\nColumn: {col}")
        print(f"Mean: {mean_val:.2f}")
        print(f"Median: {median_val:.2f}")
        print(f"Standard Deviation: {std_val:.2f}")
        print(f"Skewness Direction: {direction} (Skew: {skew_val:.2f})")

        if skew_val > 0:
            print(
                f"Interpretation: Most values are concentrated on the lower end, with a few exceptionally high values pulling the average up."
            )
        else:
            print(
                f"Interpretation: Most values are concentrated on the higher end, with a few exceptionally low values pulling the average down."
            )


def option_2():
    print("\n--- Predict Cancellation Risk ---")
    try:
        val_str = input("Enter Order Value (> 0): ")
        dist_str = input("Enter Delivery Distance in km (> 0): ")
        rating_str = input("Enter Restaurant Rating (1.0 to 5.0): ")

        val = float(val_str)
        dist = float(dist_str)
        rating = float(rating_str)

        if val <= 0 or dist <= 0:
            print(
                "Error: Order value and delivery distance must be positive numbers."
            )
            return
        if not (1.0 <= rating <= 5.0):
            print("Error: Restaurant rating must be between 1.0 and 5.0.")
            return

        input_data = pd.DataFrame([[val, dist, rating]], columns=X.columns)
        prob = rf_model.predict_proba(input_data)[0][1]
        risk = "High Cancellation Risk" if prob >= 0.5 else "Low Cancellation Risk"

        print(f"\nResult: {risk}")
        print(f"Predicted Probability of Cancellation: {prob:.2%}")

    except ValueError:
        print("Error: Invalid input. Please enter numeric values only.")


def option_3():
    print("\n--- Customer Segmentation (K-Means) ---")
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X)

    counts = pd.Series(clusters).value_counts().sort_index()
    print("\nCustomer Counts per Cluster:")
    for cluster_id, count in counts.items():
        print(f"Cluster {cluster_id}: {count} customers")

    centroids = pd.DataFrame(kmeans.cluster_centers_, columns=X.columns)
    print("\nCluster Centroids (Rounded):")
    print(centroids.round(2).to_string())


def main():
    while True:
        print("\n=== Food Delivery Customer Intelligence System ===")
        print("1. View Statistical Summary")
        print("2. Predict Cancellation Risk for a New Order")
        print("3. Run Customer Segmentation")
        print("4. Exit")

        choice = input("Select an option (1-4): ").strip()

        if choice == "1":
            option_1()
        elif choice == "2":
            option_2()
        elif choice == "3":
            option_3()
        elif choice == "4":
            print("Exiting program.")
            break
        else:
            print("Invalid choice. Please select a number between 1 and 4.")


if __name__ == "__main__":
    main()
```